In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.12.0


In [2]:
#neural network
#Input: time t
#Output: [x(t), y(t)] — predicted rabbit and fox populations

class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(), #128 i/p, 128 o/p
            nn.Linear(128, 128), nn.Tanh(), #tanh is for oscillating populations, avoids linearity
            nn.Linear(128, 2),
        )

    def forward(self, t):
        return self.net(t)


In [3]:
# physics loss - should be minimised

#parameters — same as classical solver
alpha = 1.0 #rabbit birth rate
beta  = 0.1 #predation rate when a rabbit meets fox
delta = 0.075 #fox growth rate when it eats rabbit
gamma = 1.5 #fox death rate

def physics_loss(model, t):
    t=t.clone().requires_grad_(True)   #tracks changes to t for diff
    
    out = model(t)           
    x = out[:, 0:1]          # predicted rabbits - rows and cols=0
    y = out[:, 1:2]          # predicted foxes - rows and cols=1

    # compute dx/dt and dy/dt using autograd (exact values)
    dxdt = torch.autograd.grad(x, t,
                               grad_outputs=torch.ones_like(x),
                               create_graph=True)[0]
    #ones_like sums up gradients
    dydt = torch.autograd.grad(y, t,
                               grad_outputs=torch.ones_like(y),
                               create_graph=True)[0]

    # ODE residuals — how much is the network violating the eqn?
    res_x = dxdt - (alpha*x - beta*x*y)
    res_y = dydt - (delta*x*y - gamma*y)
    #perfectly solved means it should be = 0

    return torch.mean(res_x**2) + torch.mean(res_y**2)

In [4]:
#initial condition loss

def ic_loss(model):
    t0 = torch.zeros(1, 1)   # t=0
    out = model(t0)           #initial x,y at t0
    
    x0_pred = out[0, 0]       # predicted rabbits at t=0
    y0_pred = out[0, 1]       # predicted foxes at t=0
    
    return (x0_pred - 10.0)**2 + (y0_pred - 5.0)**2

In [15]:
# creating network
model = PINN()

#after every step the nn knows which direction to take a step to reduce loss,
#so optimiser adam decides size of step (lr)
# collocation pts - where we check if ode is being satisfied 

# load classical solution as "measurement data"
ode_t       = np.load("ode_t.npy")
ode_rabbits = np.load("ode_rabbits.npy")
ode_foxes   = np.load("ode_foxes.npy")

# convert to tensors
t_data = torch.FloatTensor(ode_t).unsqueeze(1)
x_data = torch.FloatTensor(ode_rabbits).unsqueeze(1)
y_data = torch.FloatTensor(ode_foxes).unsqueeze(1)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
t_colloc = torch.linspace(0, 30, 3000).unsqueeze(1)

print("Model created!")

print("Data loaded", t_data.shape)

Model created!
Data loaded torch.Size([1000, 1])


In [17]:
print("Training PINN...")

for epoch in range(20000):
    optimizer.zero_grad()

    # data loss — match the classical solution at sampled points
    out_data  = model(t_data)
    loss_data = torch.mean((out_data[:, 0:1] - x_data)**2 + 
                           (out_data[:, 1:2] - y_data)**2)

    # physics loss — satisfy the ODE
    loss_phys = physics_loss(model, t_colloc)

    # ic loss — correct starting point
    loss_ic   = ic_loss(model)

    loss = loss_data + loss_phys + loss_ic

    loss.backward()
    optimizer.step()

    if epoch % 2000 == 0:
        print(f"Epoch {epoch:5d} | Data: {loss_data.item():.6f} | Physics: {loss_phys.item():.6f} | IC: {loss_ic.item():.6f}")

print("training done!")

Training PINN...
Epoch     0 | Data: 674.278748 | Physics: 0.223440 | IC: 130.471573
Epoch  2000 | Data: 134.267776 | Physics: 1.289659 | IC: 0.000422
Epoch  4000 | Data: 5.510040 | Physics: 0.885728 | IC: 0.001177
Epoch  6000 | Data: 0.087663 | Physics: 0.221565 | IC: 0.000061
Epoch  8000 | Data: 0.053269 | Physics: 0.111842 | IC: 0.000009
Epoch 10000 | Data: 0.025870 | Physics: 0.060330 | IC: 0.000004
Epoch 12000 | Data: 0.007386 | Physics: 0.031395 | IC: 0.000037
Epoch 14000 | Data: 0.019637 | Physics: 0.019048 | IC: 0.000076
Epoch 16000 | Data: 0.049214 | Physics: 0.025476 | IC: 0.003274
Epoch 18000 | Data: 0.000825 | Physics: 0.005441 | IC: 0.000012
training done!


In [19]:
#save prediction

t_plot = torch.linspace(0, 30, 1000).unsqueeze(1)

model.eval()
with torch.no_grad():
    pred = model(t_plot).numpy()

pinn_rabbits = pred[:, 0]
pinn_foxes   = pred[:, 1]
t_np = t_plot.squeeze().numpy()

np.save("pinn_t.npy", t_np)
np.save("pinn_rabbits.npy", pinn_rabbits)
np.save("pinn_foxes.npy", pinn_foxes)

print("Predictions saved!")

Predictions saved!
